In [1]:
import pandas as pd
df = pd.read_csv("netflix_titles.csv")
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [2]:
features = ['director', 'cast', 'listed_in', 'description']

for feature in features:
    df[feature] = df[feature].fillna('')

print(df[features].isnull().sum())

director       0
cast           0
listed_in      0
description    0
dtype: int64


In [3]:
def create_soup(x):
    return x['director'] + ' ' + x['cast'] + ' ' + x['listed_in'] + ' ' + x['description']
df['soup'] = df.apply(create_soup, axis=1)

print(df['soup'].iloc[0])

Kirsten Johnson  Documentaries As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['soup'])
print(f"Matrix shape: {tfidf_matrix.shape}")

Matrix shape: (8807, 49941)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [6]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices]

In [11]:
def clean_data(x):
    if isinstance(x, str):
        return str.lower(x.replace(" ", "").replace(",", " "))
    else:
        return ''


features = ['director', 'cast', 'listed_in']
for feature in features:
    df[feature] = df[feature].apply(clean_data)

def create_weighted_soup(x):
    return (x['director'] + ' ') * 2 + \
           (x['cast'] + ' ') + \
           (x['listed_in'] + ' ') * 3 + \
           x['description']

# Apply the new soup to our dataframe
df['soup'] = df.apply(create_weighted_soup, axis=1)

In [12]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['soup'])

from sklearn.metrics.pairwise import cosine_similarity
new_cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [13]:
exact_title = 'Spider-Man: Into the Spider-Verse'

print(f"Recommendations for {exact_title}:\n")
print(get_recommendations(exact_title, new_cosine_sim))

NEW Recommendations for Spider-Man: Into the Spider-Verse:

5846                  The Do-Over
6244                Barely Lethal
4571                 Jagga Jasoos
6077                    Abdo Mota
752               Vampire Academy
2610                    R.K.Nagar
4170                        Polar
2439              The Student Cop
3259    Iron Sky: The Coming Race
7937                  Sardaarji 2
Name: title, dtype: object
